# Afternoon class 30/08 — Worksheet 08 SOLUTIONS: alignment   (L02)

Every cell below was executed in the lab image (pandas 3.0.5) and the quoted
output is what it actually printed — including the error in Q10.

Question 5 is the one to re-read. Two labels that look identical to a human are
not identical to Pandas, and the result is a table of `NaN` with no explanation.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 08 — Alignment. Run this once.
import pandas as pd

# The lecture's two Series. Note they overlap on S2 only.
s1 = pd.Series([10, 20], index=["S1", "S2"])
s2 = pd.Series([5, 8], index=["S2", "S3"])

# Same numbers, as plain Python lists, for the comparison in Q2.
list1 = [10, 20]
list2 = [5, 8]

# Two frames that share some rows and some columns, but not all.
q1 = pd.DataFrame({"Sales": [100, 150], "Cost": [60, 90]}, index=["West", "East"])
q2 = pd.DataFrame({"Sales": [120, 200], "Tax": [12, 20]}, index=["East", "North"])

print(s1)
print()
print(s2)

PART A — matched by label, not by position

### Question 1

`s1 + s2` -> `S1 NaN`, `S2 25.0`, `S3 NaN`. -> the only shared label is `S2`.

`S2` is the one label both Series contain, so it is the one that produced
a number: 20 + 5. `S1` exists only on the left and `S3` only on the right,
so neither has both halves of an addition.

Pandas matched on the label before it added anything. The positions the
values happened to occupy were never consulted.

In [ ]:
print(s1 + s2)
print()
print("s1 labels:", list(s1.index))
print("s2 labels:", list(s2.index))
print("shared:   ", sorted(set(s1.index) & set(s2.index)))

### Question 2

Lists -> `[15, 28]`. Pandas -> two `NaN`s and one `25.0`.

The list version produced two clean integers and this is precisely the
problem. `10 + 5` is S1's value plus S2's value — two different students
added together — and `20 + 8` is S2 plus S3. Both numbers are meaningless
and neither is marked as suspect.

`zip` lined the sequences up by position because position is the only
thing a list has. The lists do not know that one starts at S1 and the
other at S2.

Pandas refused to produce those numbers at all. Two `NaN`s look like a
worse result than two integers, and they are a much better one: they are
the correct answer to 'what is S1 plus nothing'.

In [ ]:
print("lists: ", [a + b for a, b in zip(list1, list2)])
print()
print("pandas:")
print(s1 + s2)

# The list version computed 10 + 5 = 15 -- that is S1's value plus S2's
# value, two different students added together. It looks like a result.
# Pandas refused to produce that number at all.

### Question 3

Result index -> `['S1', 'S2', 'S3']`, the sorted union. -> `2` of `3` are `NaN`.

The result is not the intersection and not the left side — it is the union
of both label sets, sorted. Every label from either operand gets a row.

So an arithmetic operation can return something *longer* than both of its
inputs. Two 2-element Series produced a 3-element result. If you are
checking lengths to validate a pipeline, that is worth knowing.

In [ ]:
out = s1 + s2
print("result index:", list(out.index))
print("union of inputs:", sorted(set(s1.index) | set(s2.index)))
print()
print("NaN count:", out.isna().sum(), "of", len(out))

### Question 4

`s1.add(s2, fill_value=0)` -> `S1 10.0`, `S2 25.0`, `S3 8.0`.

Each unmatched label now takes the value from whichever side had one.
`S2` is unchanged at 25 because both sides contributed.

Note the dtype is still `float64` even though no `NaN` survives — the
promotion happens during alignment, before the fill.

And as in worksheet 05 Q4, `fill_value=0` is an assertion. Here it says
'a student with no entry in `s2` scored zero there', which is right if `s2`
is a bonus nobody was obliged to earn and wrong if `s2` is a second exam
some students have not sat yet.

In [ ]:
print("plain +:")
print(s1 + s2)
print()
print("add(fill_value=0):")
print(s1.add(s2, fill_value=0))

# S1 becomes 10 and S3 becomes 8 -- each is now "the value from whichever
# side had one". S2 is unchanged at 25 because both sides had a value.

PART B — labels that look the same and are not

### Question 5

`s1 + lower` -> **four** rows, `['S1', 'S2', 's1', 's2']`, **all `NaN`**. -> nothing raised, nothing warned.

`"S1"` and `"s1"` share no characters as far as a dictionary lookup is
concerned, so the overlap is empty and every row is unmatched. The result
is twice as long as either input and contains no data at all.

This is what a case mismatch looks like in practice, and it is worth
recognising by shape: **a result longer than both inputs, made entirely of
`NaN`.** That combination almost always means the join keys did not match,
not that your data is missing.

It happens constantly when one system exports upper-case codes and another
lower-case, and neither is wrong.

In [ ]:
lower = pd.Series([1, 1], index=["s1", "s2"])
out = s1 + lower
print(out)
print()
print("index:", list(out.index))
print("all NaN:", out.isna().all())
print("nothing raised, nothing warned")

### Question 6

Overlap before -> `set()` (empty). After `.str.upper()` -> index `['S1', 'S2']` and the sum is `S1 11`, `S2 21`.

`set(a.index) & set(b.index)` is the diagnostic. If it is empty and you
expected a join, stop and look at the labels before you look at anything
else.

Once the case matches, the addition works and the dtype stays `int64` —
no `NaN` was ever created, so nothing had to widen.

`.str` methods work on an Index the same way they work on a text column,
which is what makes the repair a one-liner.

In [ ]:
lower = pd.Series([1, 1], index=["s1", "s2"])
print("overlap before:", set(s1.index) & set(lower.index))
print()
lower.index = lower.index.str.upper()
print("index after fixing case:", list(lower.index))
print()
print(s1 + lower)

### Question 7

`s1 + padded` -> **two rows both displaying as `S1`**, both `NaN`, plus `S2 21.0`. -> `print` shows `S1`; `repr` shows `'S1 '`.

This is worse than Q5 because the output *looks* right. Two rows are
labelled `S1` on screen and there is no visible difference between them —
the trailing space is invisible in every normal print.

Only `repr()` reveals it. That is the tool: when two labels look identical
and refuse to match, `repr()` them.

Trailing whitespace arrives from spreadsheet exports, from hand-typed CSVs,
and from any system that pads fields to a fixed width. `.str.strip()` on
your keys immediately after loading costs nothing and removes an entire
category of unexplainable `NaN`.

In [ ]:
padded = pd.Series([1, 1], index=["S1 ", "S2"])
print(s1 + padded)
print()
print("printed:", padded.index[0])
print("repr:   ", repr(padded.index[0]))
print()
print("stripped:")
padded.index = padded.index.str.strip()
print(s1 + padded)

PART C — two axes at once

### Question 8

`q1 + q2` -> a 3x3 grid with **1 of 9 cells populated**: only `East`/`Sales` = `270.0`.

Alignment happened on both axes at once. The rows became the union
`East, North, West` and the columns the union `Cost, Sales, Tax`, both
sorted alphabetically — so the column order you wrote is not the column
order you get.

Only `East` appears in both indexes and only `Sales` appears in both
column sets, so exactly one cell had two values to add. Eight ninths of
your result is `NaN` from two frames that each looked complete.

The row/column union effect compounds: two 2x2 frames sharing one row and
one column produce a 3x3 result that is 89% empty.

In [ ]:
print(q1)
print()
print(q2)
print()
print(q1 + q2)
print()
total = (q1 + q2).size
present = (q1 + q2).notna().sum().sum()
print("cells:", total, "| with a value:", present)

### Question 9

`q1.add(q2, fill_value=0)` -> **7 of 9** populated. -> still missing: `('North', 'Cost')` and `('West', 'Tax')`.

`fill_value` substitutes for a value missing on *one* side. Those two
cells were missing on **both** sides — `North` never had a `Cost` in either
frame, and `West` never had a `Tax` — so there is nothing to fill from and
they stay `NaN`.

That is the correct behaviour and worth stating precisely, because
'`fill_value=0` fixes the NaNs' is a very common and slightly wrong belief.
It fixes the ones caused by *misalignment*. It cannot fix the ones caused
by *absence*.

In [ ]:
filled = q1.add(q2, fill_value=0)
print(filled)
print()
print("with a value:", filled.notna().sum().sum(), "of", filled.size)
print()
print("still missing:")
print(filled.isna().stack()[lambda s: s].index.tolist())

# West has no Tax in either frame, and North has no Cost in either.
# fill_value can substitute for ONE missing side, not for both.

### Question 10

`s1 == s2` -> **raises** `ValueError: Can only compare identically-labeled Series objects`.

Adding them was fine and comparing them is forbidden. The asymmetry is
deliberate and it is the best design decision in this part of the library.

`+` has a sensible answer for an unmatched label: `NaN`, meaning 'unknown'.
`==` does not. Is `S1` equal to a value that does not exist? `False` would
assert they differ, which you cannot know. `True` is obviously wrong.
`NaN` would break the boolean mask you are about to filter with. There is
no defensible answer, so Pandas declines to invent one.

The fix is to align explicitly first — `s1.align(s2)` or reindex both to a
common index — which forces you to decide what the missing labels mean
before you compare. Being made to make that decision is the whole point.

In [ ]:
print(s1 == s2)